# finance / portfolio / returns

Annual portfolio return to 2070 as three constants: low, base and high.

| Term | In This Notebook |
|---|---|
| **Intent** | Give one annual return per scenario to grow the portfolio to 2070. Placeholder: the base is the mean annual return and the low and high are one standard deviation either side |
| **Project Flavour** | `static`, served by `mlserver` (`model.md` 3.2) |
| **Data Splits** | `training` ten years of annual portfolio returns, synthetic until the wrangle series is loaded; `golden` the exact request that becomes `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, target, dataset names and signature names as constants (`model.md` 6.6).

In [ ]:
import json

from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "finance/portfolio/returns"
MODEL_NAME = "finance_portfolio_returns"
TARGETS = [{"kind": "served"}]
TRAINING = "portfolio_returns_training_2026-11-05_2026-11-05"
GOLDEN = "portfolio_returns_golden"
INPUT = "finance_portfolio_returns_in"
OUTPUTS = ["finance_portfolio_returns_nominal_low_pct", "finance_portfolio_returns_nominal_base_pct", "finance_portfolio_returns_nominal_high_pct"]

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training` file only when it is absent (`model.md` 6.5). The source is synthetic until the real series is loaded, so no query runs.

In [ ]:
DEVIATIONS = [-4.0, -3.0, -2.0, -1.0, 0.0, 0.0, 1.0, 2.0, 3.0, 4.0]

training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: {"year": list(range(2016, 2026)), "portfolio_return_pct": [round(6.0 + deviation, 4) for deviation in DEVIATIONS]})
print("training", training_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** take the mean and the population standard deviation of the annual returns in `training` only (R6). The low and high scenarios are one standard deviation either side of the base.

In [ ]:
BASE = round(float(training_pl["portfolio_return_pct"].mean()), 2)
SPREAD = round(float(training_pl["portfolio_return_pct"].std(ddof=0)), 2)
CONSTANTS = dict(zip(OUTPUTS, [round(BASE - SPREAD, 2), BASE, round(BASE + SPREAD, 2)], strict=True))
print("constants", CONSTANTS)
assert list(CONSTANTS.values()) == [3.55, 6.0, 8.45], "the fit did not recover the mean and spread"

## Package

- **Stage:** 5, package.
- **Intent:** write the golden rows, build the signature, write a `static` project whose constants are the fitted values, verify it and replay the golden through the library loader (R1, R3).

In [ ]:
from workbench.model import build_signature, package_project, register_preview, signature_input, signature_output
from workbench.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, lambda: {INPUT: [1.0, None], "expect": ["ok", "error"]})

signature = build_signature(
    MODEL_NAME,
    [signature_input(INPUT, "none", "Unused input.")],
    [
        signature_output("finance_portfolio_returns_nominal_low_pct", "percent", "Annual nominal portfolio return, percent per year, low scenario.", datatype="FP64"),
        signature_output("finance_portfolio_returns_nominal_base_pct", "percent", "Annual nominal portfolio return, percent per year, base scenario.", datatype="FP64"),
        signature_output("finance_portfolio_returns_nominal_high_pct", "percent", "Annual nominal portfolio return, percent per year, high scenario.", datatype="FP64"),
    ],
)
project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "static",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic annual portfolio returns, training split",
        [training_path, golden_path],
        "mean and standard deviation of annual returns",
    ),
    constants=CONSTANTS,
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped loader")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)